In [ ]:
from util import *
import math
import numpy as np
# Add your import statements here




class InformationRetrieval():

	def __init__(self):
		self.index = None
		self.total_docs = None
		self.tfidf = None
		self.words = []
		self.docIDs = None
	
	def buildIndex(self, docs, docIDs):
		"""
		Builds the document index in terms of the document
		IDs and stores it in the 'index' class variable

		Parameters
		----------
		arg1 : list
			A list of lists of lists where each sub-list is
			a document and each sub-sub-list is a sentence of the document
		arg2 : list
			A list of integers denoting IDs of the documents
		Returns
		-------
		None
		"""
		index = {}
		for doc, docID in zip(docs, docIDs):
			for sentence in doc:
				for word in sentence:
					if word not in index:
						self.words.append(word)
						index[word] = {}
						index[word][docID] = 1
					else:
						if docID not in index[word]:
							index[word][docID] = 1
						else:
							index[word][docID] += 1
		self.index = index
		self.total_docs = len(docIDs)
		self.docIDs = docIDs
		for word in index:
			self.index[word]['idf'] = math.log(self.total_docs / len(self.index[word]))
		self.tfidf = np.zeros(shape = (self.total_docs, len(self.index)))
	
		for i in range(self.total_docs):
			for j in range(len(self.index)):
				# Use .get to handle words that do not occur in a particular document
				self.tfidf[i,j] = self.index[self.words[j]]['idf'] * self.index[self.words[j]].get(docIDs[i], 0)
	
	def rank(self, queries):
		"""
		Rank the documents according to relevance for each query

		Parameters
		----------
		arg1 : list
			A list of lists of lists where each sub-list is a query and
			each sub-sub-list is a sentence of the query
		

		Returns
		-------
		list
			A list of lists of integers where the ith sub-list is a list of IDs
			of documents in their predicted order of relevance to the ith query
		"""

		doc_IDs_ordered = []
		query_tfidf = np.zeros(shape = (len(queries), len(self.words)))
		for i in range(len(queries)):
			for sentence in queries[i]:
				for word in sentence:
					if word in self.index:
						query_tfidf[i, self.words.index(word)] += self.index[word]['idf']
		normalized_query_tfidf = query_tfidf / np.linalg.norm(query_tfidf, axis = 1, keepdims = True)
		normalized_tfidf = self.tfidf / np.linalg.norm(self.tfidf, axis = 1, keepdims = True)
		cosine_similarities = np.dot(normalized_query_tfidf, normalized_tfidf.T)
		
		for i in range(len(queries)):
			doc_IDs_ordered.append([self.docIDs[j] for j in list(np.argsort(cosine_similarities[i])[::-1])])

		return doc_IDs_ordered



In [23]:
oq = None
from sentenceSegmentation import SentenceSegmentation
from tokenization import Tokenization
from inflectionReduction import InflectionReduction
from stopwordRemoval import StopwordRemoval
from informationRetrieval import InformationRetrieval
from evaluation import Evaluation

from sys import version_info
import argparse
import json
import matplotlib.pyplot as plt
import os

# Input compatibility for Python 2 and Python 3
if version_info.major == 3:
    pass
elif version_info.major == 2:
    try:
        input = raw_input
    except NameError:
        pass
else:
    print("Unknown python version - input function not safe")


class SearchEngine:

    def __init__(self, args):
        self.args = args
        self.oq = None

        # ✅ Create output folder if not exists
        if not os.path.exists(self.args.out_folder):
            os.makedirs(self.args.out_folder)

        self.tokenizer = Tokenization()
        self.sentenceSegmenter = SentenceSegmentation()
        self.inflectionReducer = InflectionReduction()
        self.stopwordRemover = StopwordRemoval()

        self.informationRetriever = InformationRetrieval()
        self.evaluator = Evaluation()

    def segmentSentences(self, text):
        if self.args.segmenter == "naive":
            return self.sentenceSegmenter.naive(text)
        elif self.args.segmenter == "punkt":
            return self.sentenceSegmenter.punkt(text)

    def tokenize(self, text):
        if self.args.tokenizer == "naive":
            return self.tokenizer.naive(text)
        elif self.args.tokenizer == "ptb":
            return self.tokenizer.pennTreeBank(text)

    def reduceInflection(self, text):
        return self.inflectionReducer.reduce(text)

    def removeStopwords(self, text):
        return self.stopwordRemover.fromList(text)

    def preprocessQueries(self, queries):
        segmentedQueries = []
        for query in queries:
            segmentedQuery = self.segmentSentences(query)
            segmentedQueries.append(segmentedQuery)

        json.dump(segmentedQueries, open(os.path.join(self.args.out_folder, "segmented_queries.txt"), 'w'))

        tokenizedQueries = []
        for query in segmentedQueries:
            tokenizedQuery = self.tokenize(query)
            tokenizedQueries.append(tokenizedQuery)

        json.dump(tokenizedQueries, open(os.path.join(self.args.out_folder, "tokenized_queries.txt"), 'w'))

        reducedQueries = []
        for query in tokenizedQueries:
            reducedQuery = self.reduceInflection(query)
            reducedQueries.append(reducedQuery)

        json.dump(reducedQueries, open(os.path.join(self.args.out_folder, "reduced_queries.txt"), 'w'))

        stopwordRemovedQueries = []
        for query in reducedQueries:
            stopwordRemovedQuery = self.removeStopwords(query)
            stopwordRemovedQueries.append(stopwordRemovedQuery)

        json.dump(stopwordRemovedQueries, open(os.path.join(self.args.out_folder, "stopword_removed_queries.txt"), 'w'))
        self.oq = stopwordRemovedQueries
        return stopwordRemovedQueries

    def preprocessDocs(self, docs):
        segmentedDocs = []
        for doc in docs:
            segmentedDoc = self.segmentSentences(doc)
            segmentedDocs.append(segmentedDoc)

        json.dump(segmentedDocs, open(os.path.join(self.args.out_folder, "segmented_docs.txt"), 'w'))

        tokenizedDocs = []
        for doc in segmentedDocs:
            tokenizedDoc = self.tokenize(doc)
            tokenizedDocs.append(tokenizedDoc)

        json.dump(tokenizedDocs, open(os.path.join(self.args.out_folder, "tokenized_docs.txt"), 'w'))

        reducedDocs = []
        for doc in tokenizedDocs:
            reducedDoc = self.reduceInflection(doc)
            reducedDocs.append(reducedDoc)

        json.dump(reducedDocs, open(os.path.join(self.args.out_folder, "reduced_docs.txt"), 'w'))

        stopwordRemovedDocs = []
        for doc in reducedDocs:
            stopwordRemovedDoc = self.removeStopwords(doc)
            stopwordRemovedDocs.append(stopwordRemovedDoc)

        json.dump(stopwordRemovedDocs, open(os.path.join(self.args.out_folder, "stopword_removed_docs.txt"), 'w'))

        return stopwordRemovedDocs

    def evaluateDataset(self):

        queries_json = json.load(open(os.path.join(args.dataset, "cran_queries.json"), 'r'))[:]
        query_ids = [item["query number"] for item in queries_json]
        queries = [item["query"] for item in queries_json]

        processedQueries = self.preprocessQueries(queries)

        docs_json = json.load(open(os.path.join(args.dataset, "cran_docs.json"), 'r'))[:]
        doc_ids = [item["id"] for item in docs_json]
        docs = [item["body"] for item in docs_json]

        processedDocs = self.preprocessDocs(docs)

        self.informationRetriever.buildIndex(processedDocs, doc_ids)
        doc_IDs_ordered = self.informationRetriever.rank(processedQueries)

        qrels = json.load(open(os.path.join(args.dataset, "cran_qrels.json"), 'r'))[:]

        precisions, recalls, fscores, MAPs, nDCGs, MRRs = [], [], [], [], [], []

        for k in range(1, 11):

            precision = self.evaluator.meanPrecision(doc_IDs_ordered, query_ids, qrels, k)
            recall = self.evaluator.meanRecall(doc_IDs_ordered, query_ids, qrels, k)
            fscore = self.evaluator.meanFscore(doc_IDs_ordered, query_ids, qrels, k)

            precisions.append(precision)
            recalls.append(recall)
            fscores.append(fscore)

            print(f"Precision, Recall, F-score @ {k}: {precision}, {recall}, {fscore}")

            MAP = self.evaluator.meanAveragePrecision(doc_IDs_ordered, query_ids, qrels, k)
            nDCG = self.evaluator.meanNDCG(doc_IDs_ordered, query_ids, qrels, k)
            MRR = self.evaluator.meanReciprocalRank(doc_IDs_ordered, query_ids, qrels, k)

            MAPs.append(MAP)
            nDCGs.append(nDCG)
            MRRs.append(MRR)

            print(f"MAP, nDCG, MRR @ {k}: {MAP}, {nDCG}, {MRR}")

        # Plot
        plt.plot(range(1, 11), precisions, label="Precision")
        plt.plot(range(1, 11), recalls, label="Recall")
        plt.plot(range(1, 11), fscores, label="F-Score")
        plt.plot(range(1, 11), MAPs, label="MAP")
        plt.plot(range(1, 11), nDCGs, label="nDCG")
        plt.plot(range(1, 11), MRRs, label="MRR")

        plt.legend()
        plt.title("Evaluation Metrics - Cranfield Dataset")
        plt.xlabel("k")
        plt.savefig(os.path.join(args.out_folder, "eval_plot.png"))

    def handleCustomQuery(self):

        print("Enter query below")
        query = input()

        processedQuery = self.preprocessQueries([query])[0]

        docs_json = json.load(open(os.path.join(args.dataset, "cran_docs.json"), 'r'))[:]
        doc_ids = [item["id"] for item in docs_json]
        docs = [item["body"] for item in docs_json]

        processedDocs = self.preprocessDocs(docs)

        self.informationRetriever.buildIndex(processedDocs, doc_ids)
        doc_IDs_ordered = self.informationRetriever.rank([processedQuery])[0]

        print("\nTop five document IDs : ")
        for id_ in doc_IDs_ordered[:5]:
            print(id_)


# if __name__ == "__main__":

#     parser = argparse.ArgumentParser(description='main.py')

    # parser.add_argument('-dataset', default="cranfield/")
    # parser.add_argument('-out_folder', default="output/")
    # parser.add_argument('-segmenter', default="punkt")
    # parser.add_argument('-tokenizer', default="ptb")
    # parser.add_argument('-custom', action="store_true")

#     args = parser.parse_args()

#     searchEngine = SearchEngine(args)

#     if args.custom:
#         searchEngine.handleCustomQuery()
#     else:
#         searchEngine.evaluateDataset()


In [24]:
args.custom

False

In [25]:
parser = argparse.ArgumentParser(description='main.py')

parser.add_argument('-dataset', default="cranfield/")
parser.add_argument('-out_folder', default="output/")
parser.add_argument('-segmenter', default="punkt")
parser.add_argument('-tokenizer', default="ptb")
parser.add_argument('-custom', action="store_true")

# Notebook-safe: ignore unknown Jupyter kernel args
args, _ = parser.parse_known_args()

searchEngine = SearchEngine(args)

searchEngine.handleCustomQuery()

# if args.custom:
#     searchEngine.handleCustomQuery()
# else:
#     searchEngine.evaluateDataset()


Enter query below

Top five document IDs : 
471
995
325
111
382


/home/crimson/Projects/Acads/NLP/Project/assignment2/informationRetrieval.py:85: RuntimeWarning: invalid value encountered in divide
  normalized_tfidf = self.tfidf / np.linalg.norm(self.tfidf, axis = 1, keepdims = True)


In [27]:
query = searchEngine.oq

In [28]:
ir = searchEngine.informationRetriever

In [42]:
eps = 1e-10

In [43]:
words = ir.words
index = ir.index
normalized_tfidf = ir.tfidf / (eps + np.linalg.norm(ir.tfidf, axis = 1, keepdims = True))
docIDs = ir.docIDs

In [44]:
import numpy as np

In [45]:
doc_IDs_ordered = []
query_tfidf = np.zeros(shape = (len(query), len(words)))
for i in range(len(query)):
    for sentence in query[i]:
        for word in sentence:
            if word in index:
                query_tfidf[i, words.index(word)] += index[word]['idf']
normalized_query_tfidf = query_tfidf / np.linalg.norm(query_tfidf, axis = 1, keepdims = True)
cosine_similarities = np.dot(normalized_query_tfidf, normalized_tfidf.T)

for i in range(len(query)):
    doc_IDs_ordered.append([docIDs[j] for j in list(np.argsort(cosine_similarities[i])[::-1])])

In [47]:
np.sum(cosine_similarities)

np.float64(10.933148562390127)

In [48]:
doc_IDs_ordered

[[325,
  111,
  382,
  92,
  1063,
  1006,
  493,
  231,
  248,
  127,
  1087,
  349,
  1388,
  677,
  945,
  1041,
  774,
  901,
  425,
  457,
  322,
  1043,
  161,
  537,
  745,
  211,
  1023,
  375,
  266,
  848,
  93,
  1246,
  383,
  1253,
  637,
  684,
  1328,
  1233,
  321,
  1088,
  648,
  1183,
  1005,
  1123,
  48,
  1055,
  818,
  1071,
  1300,
  770,
  359,
  14,
  1248,
  1112,
  16,
  1286,
  279,
  410,
  498,
  1275,
  86,
  1110,
  1222,
  869,
  249,
  1175,
  1390,
  771,
  783,
  606,
  489,
  1298,
  150,
  504,
  374,
  949,
  1217,
  1385,
  543,
  417,
  1007,
  433,
  985,
  1294,
  1193,
  118,
  827,
  733,
  552,
  1194,
  987,
  980,
  924,
  32,
  689,
  328,
  113,
  762,
  59,
  317,
  443,
  64,
  1343,
  730,
  282,
  1225,
  1280,
  790,
  563,
  24,
  49,
  782,
  1019,
  704,
  1061,
  85,
  499,
  1195,
  476,
  225,
  1072,
  927,
  427,
  458,
  460,
  487,
  486,
  377,
  461,
  462,
  463,
  376,
  464,
  459,
  488,
  465,
  379,
  366,
  380,